In [0]:
# TODOS
# 1. Centralize the write and read methods into a single notebook to be reused
# 2. Unit Test & Integration Test

import os
from pyspark.sql import functions as F
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, ShortType
)

# ── Paths ──────────────────────────────────────────────────────────────────

BASE_DIR = '/Volumes/debora_ryan_susheela_hhs/default/upload_volume'
ALL_DATA_DIR = os.path.join(BASE_DIR, "data", "all_data_hhs")
BRONZE_PARQUET_DIR = os.path.join(BASE_DIR, "bronze_output", "parquet_data_hhs")

os.makedirs(ALL_DATA_DIR, exist_ok=True)
os.makedirs(BRONZE_PARQUET_DIR, exist_ok=True)
print(f"BASE_DIR: {BASE_DIR}")
print(f"ALL_DATA_DIR: {ALL_DATA_DIR}")
print(f"BRONZE_PARQUET_DIR: {BRONZE_PARQUET_DIR}")

GLOSSARY
- PlanId : Unique plan identifier (14 chars e.g. 21989AK0010001). Primary join key to BenefitsCostSharing.StandardComponentId.
- IndividualRate: Monthly premium in $ for one enrollee. The central measure in all premium analyses.
- Age: Age bracket of the enrollee. Values: 0-20 / 21 through 64 / 65 and over / Family Option. Requires parsing before aggregation.
- Tobacco: Tobacco use status. Values: No Preference (non-tobacco) or Tobacco. Insurers can charge up to 1.5x for tobacco users.
- RatingAreaId : Geographic pricing zone within a state (e.g. Rating Area 1). Premiums vary by rating area even for the same plan.
- StateCode: Two-letter US state code.
- BusinessYear: Plan year (2014 / 2015 / 2016).
- RateEffectiveDate,Start date of this rate row : typically Jan 1 of BusinessYear.
- RateExpirationDate,End date of this rate row: typically Dec 31 of BusinessYear.
- IssuerId: Unique identifier for the insurance company offering the plan.
- IndividualTobaccoRate: Monthly premium for tobacco users. Null when the plan does not distinguish tobacco rates separately.
- Couple: Monthly rate for a two-adult enrollment.
- BenefitName: Name of the covered service (e.g. Primary Care Visit to Treat an Injury or Illness / Generic Drugs). The filter key for any service-specific analysis.
- CopayInnTier1: In-network Tier 1 flat copay — free-text string (e.g. $30 Copay before deductible / No Charge). Requires regex parsing to extract a dollar amount.
- CopayInnTier2: In-network Tier 2 copay. Only present in plans with a two-tier network (preferred vs. standard providers).
- CopayOutofNet,Out-of-network flat copay. Usually higher than Tier 1.
- CoinsInnTier1,In-network Tier 1 coinsurance — percentage string (e.g. 20%). Patient pays this share of the bill after deductible. Cannot be compared directly to a flat copay dollar.
- CoinsInnTier2,Tier 2 coinsurance rate. Only populated in two-tier network plans.
- CoinsOutofNet: Out-of-network coinsurance percentage.
- IsCovered: Whether the service is covered at all (Covered / Not Covered). Filter to Covered before any cost analysis.


In [0]:
# SCHEMA Defintions for BenefitsCost Sharing and Rates csv's

BENEFITS_SCHEMA = StructType([
    StructField("BenefitName",       StringType(), True),
    StructField("BusinessYear",      ShortType(),  True),
    StructField("CoinsInnTier1",     StringType(), True),
    StructField("CoinsInnTier2",     StringType(), True),
    StructField("CoinsOutofNet",     StringType(), True),
    StructField("CopayInnTier1",     StringType(), True),
    StructField("CopayInnTier2",     StringType(), True),
    StructField("CopayOutofNet",     StringType(), True),
    StructField("EHBVarReason",      StringType(), True),
    StructField("Exclusions",        StringType(), True),
    StructField("Explanation",       StringType(), True),
    StructField("ImportDate",        StringType(), True),
    StructField("IsCovered",         StringType(), True),
    StructField("IsEHB",             StringType(), True),
    StructField("IsExclFromInnMOOP", StringType(), True),
    StructField("IsExclFromOonMOOP", StringType(), True),
    StructField("IsStateMandate",    StringType(), True),
    StructField("IsSubjToDedTier1",  StringType(), True),
    StructField("IsSubjToDedTier2",  StringType(), True),
    StructField("IssuerId",          StringType(), True),
    StructField("IssuerId2",         StringType(), True),
    StructField("LimitQty",          StringType(), True),
    StructField("LimitUnit",         StringType(), True),
    StructField("MinimumStay",       StringType(), True),
    StructField("PlanId",            StringType(), True),
    StructField("QuantLimitOnSvc",   StringType(), True),
    StructField("RowNumber",         IntegerType(), True),
    StructField("SourceName",        StringType(), True),
    StructField("StandardComponentId",StringType(),True),
    StructField("StateCode",         StringType(), True),
    StructField("StateCode2",        StringType(), True),
    StructField("VersionNum",        IntegerType(), True),
])

# ── Rate ───────────────────────────────────────────────────────────────────
RATE_SCHEMA = StructType([
    StructField("BusinessYear",                              ShortType(),  True),
    StructField("StateCode",                                StringType(), True),
    StructField("IssuerId",                                 StringType(), True),
    StructField("SourceName",                               StringType(), True),
    StructField("VersionNum",                               IntegerType(), True),
    StructField("ImportDate",                               StringType(), True),
    StructField("IssuerId2",                                StringType(), True),
    StructField("FederalTIN",                               StringType(), True),
    StructField("RateEffectiveDate",                        StringType(), True),
    StructField("RateExpirationDate",                       StringType(), True),
    StructField("PlanId",                                   StringType(), True),
    StructField("RatingAreaId",                             StringType(), True),
    StructField("Tobacco",                                  StringType(), True),
    StructField("Age",                                      StringType(), True),
    StructField("IndividualRate",                           StringType(), True),
    StructField("IndividualTobaccoRate",                    StringType(), True),
    StructField("Couple",                                   StringType(), True),
    StructField("PrimarySubscriberAndOneDependent",         StringType(), True),
    StructField("PrimarySubscriberAndTwoDependents",        StringType(), True),
    StructField("PrimarySubscriberAndThreeOrMoreDependents",StringType(), True),
    StructField("CoupleAndOneDependent",                   StringType(), True),
    StructField("CoupleAndTwoDependents",                  StringType(), True),
    StructField("CoupleAndThreeOrMoreDependents",          StringType(), True),
    StructField("RowNumber",                                IntegerType(), True),
])

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType

def read_data_from_csv(filepath: str, schema: StructType) -> DataFrame:
    """Read a CSV with an explicit schema."""
    reader = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("delimiter", ",")
        .option("escape", '"')
        .option("multiLine", True)
        .schema(schema)
    )
    df = reader.load(filepath)
    return df

In [0]:
# Write data to parquet
def write(input_df: DataFrame, out_dir):
    return input_df.write.mode('overwrite').parquet(out_dir)

In [0]:
# Write data to parquet for each file. TODO: Mask the ITIN here.
bronze = {
    "benefits":     read_data_from_csv(f"{BASE_DIR}/BenefitsCostSharing.csv",  BENEFITS_SCHEMA),
    "rates":        read_data_from_csv(f"{BASE_DIR}/Rate.csv",  RATE_SCHEMA)
}

# Mask FederalTIN in rates
bronze["rates"] = bronze["rates"].withColumn(
    "FederalTIN",
    F.when(F.col("FederalTIN").isNotNull(), F.sha2(F.col("FederalTIN"), 256)).otherwise(None)
)

print("Bronze DataFrames loaded:")
for name, df in bronze.items():
    write(df, f"{BRONZE_PARQUET_DIR}/{name}")
    print(f"  {name}  {df.count()}  ...")

# Unit Tests

In [0]:
# # Install a few helpers we prepared for you
%pip uninstall -y databricks_helpers exercise_ev_databricks_unit_tests

# # Install the databricks helpers 
# # %pip install git+https://github.com/data-derp/databricks_helpers.git@sr/dbr_17.3_lts_testing
%pip install git+https://github.com/data-derp/databricks_helpers.git

# # # Install the databricks test cases
# # %pip install git+https://github.com/data-derp/exercise_ev_databricks_unit_tests.git@sr/dbr_17.3_lts_testing
%pip install git+https://github.com/data-derp/exercise_ev_databricks_unit_tests.git

In [0]:
from exercise_ev_databricks_unit_tests.batch_processing_bronze import test_write_e2e

# Validate if the bronze data was written correctly
test_write_e2e(dbutils.fs.ls(f"{BRONZE_PARQUET_DIR}/benefits"), spark, display)
test_write_e2e(dbutils.fs.ls(f"{BRONZE_PARQUET_DIR}/rates"), spark, display)